# Fundamentals 00.4 - Runtime vLLM Provider API

Objetivo: probar la ruta `vllm-runtime` de forma aislada antes de usar agentes, systems o graphs con un modelo local/GPU.

Este notebook ense?a la capa provider para vLLM. vLLM no es framework: es infraestructura externa que expone una API OpenAI-compatible. Agentic Systems se conecta a esa API con `provider="vllm-runtime"`.

Regla de dise?o:

```text
Agentic Systems define el contrato de ejecuci?n.
vllm-runtime define el backend OpenAI-compatible.
vLLM server corre fuera de la librer?a, normalmente en Colab/GPU.
```


## 0) Imports m?nimos

El notebook asume que `agentic-systems` est? instalado en el ambiente activo. Para usar el cliente vLLM necesitas tambi?n el extra OpenAI:

```bash
pip install -e ".[openai]"
```

En Colab/PyPI ser? equivalente a instalar el paquete y el cliente OpenAI. El servidor GPU de vLLM se instala y arranca aparte.


In [ ]:
import json
import os
from urllib.request import Request, urlopen

import agentic_systems as toolkit

print("agentic_systems:", toolkit.__name__)


## 1) Configuraci?n del provider

`vllm-runtime` lee configuraci?n desde variables de entorno o `.env`:

| Variable | Uso |
|---|---|
| `VLLM_BASE_URL` | URL OpenAI-compatible del servidor vLLM. |
| `VLLM_API_BASE` | Alias alternativo para la URL. |
| `AGENTIC_SYSTEMS_VLLM_BASE_URL` | URL espec?fica para Agentic Systems. |
| `VLLM_MODEL_ID` | Modelo servido por vLLM. |
| `VLLM_MODEL` | Alias alternativo de modelo. |
| `AGENTIC_SYSTEMS_VLLM_MODEL_ID` | Modelo espec?fico para Agentic Systems. |
| `VLLM_API_KEY` | API key para servidores compatibles; normalmente `EMPTY` local. |

El default local es `http://127.0.0.1:8000/v1` y modelo `Qwen/Qwen3-0.6B`. El notebook no guarda secretos.


In [ ]:
# Valores seguros para Colab/local si todav?a no est?n definidos.
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
os.environ.setdefault("VLLM_MODEL_ID", "Qwen/Qwen3-0.6B")
os.environ.setdefault("VLLM_API_KEY", "EMPTY")

vllm_env = {
    "VLLM_BASE_URL": os.getenv("VLLM_BASE_URL"),
    "VLLM_MODEL_ID": os.getenv("VLLM_MODEL_ID"),
    "VLLM_API_KEY_configured": bool(os.getenv("VLLM_API_KEY")),
}

toolkit.show(vllm_env, title="Configuraci?n vLLM segura")


## 2) Declarar `RuntimeConfig`

`toolkit.runtime(provider="vllm-runtime")` no ejecuta el modelo. Solo declara el contrato de provider, modelo, scheduler y metadata segura.

`runtime.describe()` sirve para confirmar qu? leer? Agentic Systems antes de hacer inferencia.


In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=0,
    max_tool_calls=4,
    max_turns=4,
    max_concurrency=1,
)

vllm_runtime = toolkit.runtime(
    provider="vllm-runtime",
    scheduler=scheduler,
)

auto_runtime = toolkit.runtime(
    provider="auto",
    scheduler=scheduler,
)

toolkit.show(vllm_runtime.describe(), title="vLLM runtime - describe")
toolkit.show(auto_runtime.describe(), title="Auto runtime - describe")


## 3) Health check opcional del servidor

Esta celda intenta consultar `/models` en el servidor vLLM OpenAI-compatible. Si no hay servidor levantado, no falla el notebook: reporta `skipped`.

En Colab, primero levanta vLLM con un comando equivalente a:

```bash
python -m vllm.entrypoints.openai.api_server   --model Qwen/Qwen3-0.6B   --served-model-name Qwen/Qwen3-0.6B   --host 127.0.0.1   --port 8000
```


In [ ]:
def vllm_models_url(base_url: str) -> str:
    return base_url.rstrip("/") + "/models"

models_url = vllm_models_url(os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8000/v1"))

try:
    request = Request(models_url, headers={"Authorization": f"Bearer {os.getenv('VLLM_API_KEY', 'EMPTY')}"})
    with urlopen(request, timeout=3) as response:
        payload = json.loads(response.read().decode("utf-8"))
    toolkit.show({"status": "ok", "models_url": models_url, "response": payload}, title="vLLM server health")
    VLLM_SERVER_AVAILABLE = True
except Exception as exc:
    toolkit.show({"status": "skipped", "models_url": models_url, "reason": str(exc)}, title="vLLM server health")
    VLLM_SERVER_AVAILABLE = False


## 4) Tool smoke opcional con `vllm-runtime`

Este smoke usa una tool normal de Agentic Systems. Si el servidor vLLM est? disponible y soporta tool calling compatible, el agente puede llamarla.

Si el servidor no est? levantado, la celda reporta `skipped` para que el notebook siga siendo portable en local, VSCode y Colab.


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos enteros."""
    return {"result": a + b}

policy = toolkit.RunPolicy(
    max_turns=4,
    max_tool_calls=2,
    temperature=0.0,
    tool_choice="auto",
    repair=True,
    max_repairs=1,
    trace="compact",
    strict=True,
)

if not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="Tool smoke vLLM")
    result = None
else:
    system = toolkit.AgenticSystem(model=vllm_runtime.model_id or "Qwen/Qwen3-0.6B", runtime=vllm_runtime)
    agent = system.agent(
        name="qwen_calculator",
        instructions="Usa la tool sumar para resolver la petici?n y responde breve.",
        tools=[sumar],
        engine="vllm-runtime",
        runtime=vllm_runtime,
        policy=policy,
    )
    result = agent.run("Suma 10 y 20 usando la tool sumar.", mode="eval")
    toolkit.human_result(result)


## 5) Cierre de API

Este notebook cubre la ruta provider vLLM sin introducir frameworks externos. La composici?n con `AgenticSystem`, LangGraph, Strands u OpenAI Agents se ense?a en notebooks posteriores.


In [ ]:
api_coverage = [
    {"api": "toolkit.runtime(provider='vllm-runtime')", "description": "Declara vLLM como provider canonico."},
    {"api": "toolkit.runtime(provider='auto')", "description": "Selecciona vLLM automaticamente cuando VLLM_BASE_URL esta configurado."},
    {"api": "RuntimeConfig.describe", "description": "Muestra resolucion y configuracion segura."},
    {"api": "toolkit.scheduler", "description": "Declara limites de ejecucion."},
    {"api": "toolkit.RunPolicy", "description": "Controla loops, tool calls y trazas."},
    {"api": "toolkit.human_result", "description": "Renderiza resultados reales de ejecucion."},
]

toolkit.show({"notebook": "00_runtime_vllm_provider_api.ipynb", "api_coverage": api_coverage}, title="API coverage")
